# EarthLens AI | Colab quick start
Choose **Runtime → Change runtime type → T4 GPU**. Run the quick-test cells first. Smoke results are not final results. Full training cells are optional until ready. Upload the revised ZIP when asked.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall('/content')
os.chdir('/content/EarthLens_AI')
print(Path.cwd())

In [ ]:
%pip install -q -r requirements.txt
!apt-get -qq install graphviz

In [ ]:
import subprocess, sys
from pathlib import Path
import tensorflow as tf
print('TensorFlow:', tf.__version__, 'GPU:', tf.config.list_physical_devices('GPU'))

def run(*args):
    subprocess.run([sys.executable, *map(str,args)], check=True)

run('data/download_data.py')
run('-m','src.data','--audit','--split-report')

## Quick test — check the pipeline, no final claims
The following cells use capped data and random ResNet weights to avoid a pretrained download. These runs do not qualify for final model selection.

In [ ]:
run('-m','src.train','--model','model_a','--smoke','--run-name','quick_a','--author','Quick check')

In [ ]:
run('-m','src.train','--model','model_b','--smoke','--no-pretrained','--run-name','quick_b','--author','Quick check')
run('-m','src.train','--model','model_c','--smoke','--no-pretrained','--parent-run','quick_b','--run-name','quick_c','--author','Quick check')

In [ ]:
from configs import config as cfg
sample = next((cfg.DATA_DIR / 'River').glob('*.jpg'))
run('-m','src.predict','--model','models/quick_a.keras','--image',sample)
run('-m','src.models','--summary','model_a','--hand-count','--no-pretrained')

## Full training (only after quick test)
Run these cells separately. Use your own author names. Training needs GPU and time. Do not use smoke metrics in the report. The EDA notebook in `notebooks/01_eda.ipynb` should also be run and saved with outputs.

In [ ]:
# Run full EDA notebook and save outputs.
import nbformat
from nbclient import NotebookClient
eda = nbformat.read('notebooks/01_eda.ipynb', as_version=4)
NotebookClient(eda, timeout=600, resources={'metadata': {'path': str(Path.cwd())}}).execute()
nbformat.write(eda, 'notebooks/01_eda.ipynb')

In [ ]:
run('-m','src.baselines','--epochs','12')
run('-m','src.train','--model','model_a','--author','Member2')

In [ ]:
run('-m','src.train','--model','model_b','--author','Member3')
run('-m','src.train','--model','model_c','--author','Member3')
run('-m','src.models','--tape-demo','--steps','20')

## Controlled tuning and ablations
Run each study once. Update author names to match actual contributors. All arms use validation only.

In [ ]:
for study in ('lr','batch','optimizer','architecture'):
    run('-m','src.run_experiments','--study',study,'--author','Member4')

In [ ]:
for study, author in [('ablation_aug','Member1'),('ablation_weights','Member1'),('ablation_dropout','Member3'),('ablation_bn','Member3')]:
    run('-m','src.run_experiments','--study',study,'--author',author)

In [ ]:
run('-m','src.run_experiments','--study','seeds','--author','Tech Lead')

## Final test — only once after freezing selection
Do not run this cell until all tuning is complete. Inspect the real errors and write your report.

In [ ]:
run('-m','src.evaluate','--latest','--all-models')
run('-m','src.predict','--latest','--image',sample)

## Save your artifacts
Save before the Colab session ends. This excludes raw EuroSAT data and downloaded archive.

In [ ]:
import zipfile
with zipfile.ZipFile('/content/EarthLens_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for folder in ('experiments','models','notebooks'):
        for p in Path(folder).rglob('*'):
            if p.is_file(): z.write(p,p.as_posix())
files.download('/content/EarthLens_results.zip')